<a href="https://colab.research.google.com/github/jaysulk/GENERIC-FNO/blob/main/GENERIC_FNO_4_PDE_noIN_Experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
# ============================================================================
# GENERIC-FNO : InstanceNorm ablation runner  (NEW FILE -- leaves repo unchanged)
# ============================================================================
# Adds one model to the paper's own 1D-FNO and 2D-FNO benchmarks:
#
#     GENERIC-noIN  =  GENERIC-FNO with every InstanceNorm inside the E,S
#                      functional backbones replaced by Identity AT INIT
#                      (i.e. trained without it; identical everywhere else).
#
# Everything else -- generators, models, train_model, evaluate_model,
# run_benchmark, run_benchmark_seeds, channel_diagnostics, the 80/20 split, the
# once-at-top seeding, the checkpointing -- is copied verbatim from the two repo
# scripts so the numbers are directly comparable to Table 1 / Table 7.  The only
# code changes are: (1) the strip_instance_norm helper, (2) the extra model in
# each run_benchmark's model list, (3) a checkpoint-filename fix so the two
# GENERIC variants do not overwrite each other's .pt.
#
# Run in Colab:  set DRIVE below, then the __main__ block runs 1D (5 seeds) and
# 2D (3 seeds, heat/advection/burgers).  Env overrides: R_1D_SEEDS, R_2D_SEEDS,
# R_2D_NX, R_2D_EPOCHS, R_RUN ("1d","2d","both").
# ============================================================================
import os, time, math, pickle
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


def strip_instance_norm(model):
    """Replace every InstanceNorm inside model.E_net / model.S_net with Identity,
    in place. Operators, projections, heads, and the residual path are untouched.
    Applied at init (before training), so the model is *trained* without IN --
    a post-hoc strip of a trained model would be meaningless."""
    n = 0
    for net in (model.E_net, model.S_net):
        for mod in list(net.modules()):
            for cname, child in list(mod.named_children()):
                if isinstance(child, (nn.InstanceNorm1d, nn.InstanceNorm2d, nn.InstanceNorm3d)):
                    setattr(mod, cname, nn.Identity()); n += 1
    assert n > 0, "no InstanceNorm found in E_net/S_net -- wrong model class?"
    return model


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================================
# ============================  1D  FNO  SECTION  ============================
# ============================================================================
# (generators, SpectralConv1d/FNO_Block/FNO_Backbone/FunctionalNet, VanillaFNO,
#  EP_FNO, GENERIC_FNO, train_model -- copied verbatim from the repo 1D script)


def generate_heat_data(n_samples=200, nx=64, nt=20, dt=0.01, nu=0.01):
    """Heat equation: du/dt = nu * d²u/dx². Purely dissipative."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    decay = torch.exp(-nu * k**2 * dt)

    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, nx//4, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            phase = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + phase)

        u_hat = torch.fft.rfft(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft(u_hat, n=nx))

        traj = torch.stack(traj, dim=0)  # (nt+1, nx)
        data_in.append(traj[:-1])
        data_out.append(traj[1:])

    return torch.stack(data_in), torch.stack(data_out), 'heat'


def generate_wave_data(n_samples=200, nx=64, nt=20, dt=0.01, c=1.0):
    """Wave equation (1st order system). Purely reversible. We track u-component only."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    omega = c * k
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx); v0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, nx//4, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            phase = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + phase)
            amp_v = torch.randn(1).item() * 0.3
            v0 += amp_v * torch.cos(ki * x + phase)
        u_hat = torch.fft.rfft(u0); v_hat = torch.fft.rfft(v0)
        traj = [u0.clone()]
        for t in range(nt):
            cos_w = torch.cos(omega * dt); sin_w = torch.sin(omega * dt)
            u_new = cos_w * u_hat + 1j * sin_w * v_hat
            v_new = 1j * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft(u_hat, n=nx))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'wave'


def generate_advection_data(n_samples=200, nx=64, nt=20, dt=0.01, c=1.0, max_mode=6):
    """1D linear advection: du/dt + c*du/dx = 0. Reversible, band-limited to max_mode."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    phase = torch.exp(-1j * c * k * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 6, (1,)).item()
        u0 = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, max_mode+1, (1,)).item()
            amp = torch.randn(1).item() * 0.5
            ph = torch.rand(1).item() * 2 * math.pi
            u0 += amp * torch.sin(ki * x + ph)
        u_hat = torch.fft.rfft(u0)
        traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft(u_hat, n=nx))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'


def generate_burgers_data(n_samples=200, nx=64, nt=20, dt=0.005, nu=0.02):
    """Viscous Burgers: du/dt + u*du/dx = nu*d²u/dx². Mixed rev+diss."""
    x = torch.linspace(0, 2*math.pi, nx+1)[:-1]
    k = torch.fft.rfftfreq(nx, d=1.0/nx) * 2 * math.pi / (2*math.pi)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = torch.zeros(nx)
        for __ in range(n_modes):
            ki = torch.randint(1, 5, (1,)).item()
            amp = torch.randn(1).item() * 0.3
            phase = torch.rand(1).item() * 2 * math.pi
            u += amp * torch.sin(ki * x + phase)
        traj = [u.clone()]
        for t in range(nt):
            u_hat = torch.fft.rfft(u)
            u_hat = u_hat / (1 + nu * k**2 * dt)
            u = torch.fft.irfft(u_hat, n=nx)
            du_dx = torch.fft.irfft(1j * k * torch.fft.rfft(u), n=nx)
            u = u - dt * u * du_dx
            traj.append(u.clone())
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'burgers'


class SpectralConv1d(nn.Module):
    """Standard FNO spectral convolution (1D). Full-rank."""
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        self.modes = modes
        scale = 1.0 / (in_ch * out_ch)
        self.W = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes, dtype=torch.cfloat))

    def forward(self, x):
        B, C, N = x.shape
        x_hat = torch.fft.rfft(x, dim=-1)
        m = min(self.modes, x_hat.shape[-1])
        out_hat = torch.zeros(B, self.W.shape[0], x_hat.shape[-1],
                              dtype=torch.cfloat, device=x.device)
        out_hat[:, :, :m] = torch.einsum('bix,oix->box', x_hat[:, :, :m], self.W[:, :, :m])
        return torch.fft.irfft(out_hat, n=N)


class FNO_Block(nn.Module):
    """Single FNO layer: spectral conv + skip + activation."""
    def __init__(self, width, modes):
        super().__init__()
        self.conv = SpectralConv1d(width, width, modes)
        self.skip = nn.Conv1d(width, width, 1)
        self.norm = nn.InstanceNorm1d(width)

    def forward(self, x):
        return F.gelu(self.norm(self.conv(x) + self.skip(x)))


class FNO_Backbone(nn.Module):
    """Multi-layer FNO backbone: lift → N layers → output channels."""
    def __init__(self, in_ch=1, out_ch=1, width=32, modes=16, n_layers=4):
        super().__init__()
        self.lift = nn.Conv1d(in_ch, width, 1)
        self.blocks = nn.ModuleList([FNO_Block(width, modes) for _ in range(n_layers)])
        self.proj = nn.Sequential(
            nn.Conv1d(width, width, 1),
            nn.GELU(),
            nn.Conv1d(width, out_ch, 1)
        )

    def forward(self, x):
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.proj(x)


class FunctionalNet(nn.Module):
    """FNO backbone → scalar functional F[u]."""
    def __init__(self, width=24, modes=12, n_layers=3):
        super().__init__()
        self.backbone = FNO_Backbone(in_ch=1, out_ch=1, width=width,
                                      modes=modes, n_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(1, 16),
            nn.GELU(),
            nn.Linear(16, 1)
        )

    def forward(self, u):
        density = self.backbone(u)           # (B, 1, N)
        integral = density.mean(dim=-1)      # (B, 1)
        return self.head(integral).squeeze(-1)  # (B,)


class VanillaFNO(nn.Module):
    """Standard FNO with residual skip: u_next = u + FNO(u)."""
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone(in_ch=1, out_ch=1, width=width,
                                      modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        u_next = self.forward(u)
        return u_next, {}


class EP_FNO(nn.Module):
    """FNO + energy penalty loss. Soft physics constraint."""
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone(in_ch=1, out_ch=1, width=width,
                                      modes=modes, n_layers=n_layers)

    def forward(self, u):
        return u + self.backbone(u)

    def predict_with_info(self, u):
        u_next = self.forward(u)
        E_in = 0.5 * (u**2).mean(dim=-1).mean(dim=-1)
        E_out = 0.5 * (u_next**2).mean(dim=-1).mean(dim=-1)
        dE = E_out - E_in
        return u_next, {'energy_in': E_in, 'energy_out': E_out, 'dE': dE}

    def energy_penalty(self, info, pde_type):
        dE = info['dE']
        if pde_type in ('heat', 'burgers'):
            return (F.relu(dE)**2).mean()
        elif pde_type in ('wave', 'advection'):
            return (dE**2).mean()
        else:
            return torch.tensor(0.0, device=dE.device)


class GENERIC_FNO(nn.Module):
    """du/dt = L dE/du + M dS/du with degeneracy by construction."""

    def __init__(self, nx=64, width_func=24, modes_func=12, n_layers_func=3,
                 modes_op=16, degeneracy_construction=True, l2_vargrad=False,
                 use_residual=True):
        super().__init__()
        self.nx = nx
        self.modes_op = modes_op
        self.degeneracy_construction = degeneracy_construction
        self.l2_vargrad = l2_vargrad
        self.use_residual = use_residual
        n_rfft = nx // 2 + 1
        m = min(modes_op, n_rfft)
        self.m = m

        self.E_net = FunctionalNet(width=width_func, modes=modes_func,
                                    n_layers=n_layers_func)
        self.S_net = FunctionalNet(width=width_func, modes=modes_func,
                                    n_layers=n_layers_func)

        self.a = nn.Parameter(0.3 * torch.randn(m))
        self.b_real = nn.Parameter(0.3 * torch.randn(m))
        self.b_imag = nn.Parameter(0.3 * torch.randn(m))

        self.residual = nn.Sequential(
            nn.Conv1d(1, 16, 1),
            nn.GELU(),
            nn.Conv1d(16, 1, 1)
        )
        self.residual_gate = nn.Parameter(torch.tensor(-3.0))

    def _get_operators(self):
        L_k = 1j * self.a
        M_k = self.b_real**2 + self.b_imag**2
        return L_k, M_k

    def _L_apply(self, v):
        N = v.shape[-1]; m = self.m
        vh = torch.fft.rfft(v, dim=-1)
        out = torch.zeros_like(vh)
        out[:, :, :m] = (1j * self.a) * vh[:, :, :m]
        return torch.fft.irfft(out, n=N)

    def _M_apply(self, v):
        N = v.shape[-1]; m = self.m
        vh = torch.fft.rfft(v, dim=-1)
        out = torch.zeros_like(vh)
        M_k = self.b_real**2 + self.b_imag**2
        out[:, :, :m] = M_k * vh[:, :, :m]
        return torch.fft.irfft(out, n=N)

    @staticmethod
    def _remove(v, w):
        """(I - P_w) v: remove component of v along direction w, per sample."""
        ip = (v * w).sum(dim=-1, keepdim=True)
        nn_ = (w * w).sum(dim=-1, keepdim=True) + 1e-12
        return v - (ip / nn_) * w

    def _generic_rhs(self, dEdu, dSdu):
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu)), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu)), dEdu)
        return rev, diss

    def forward(self, u):
        B, C, N = u.shape
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]

        if self.l2_vargrad:
            scale = N / (2.0 * math.pi)
            dEdu = dEdu * scale
            dSdu = dSdu * scale

        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu)
            return u + rev + diss

        L_k, M_k = self._get_operators()
        m = L_k.shape[0]
        dEdu_hat = torch.fft.rfft(dEdu, dim=-1)
        dSdu_hat = torch.fft.rfft(dSdu, dim=-1)
        rev_hat = torch.zeros_like(dEdu_hat)
        rev_hat[:, :, :m] = L_k.unsqueeze(0).unsqueeze(0) * dEdu_hat[:, :, :m]
        rev = torch.fft.irfft(rev_hat, n=N)
        diss_hat = torch.zeros_like(dSdu_hat)
        diss_hat[:, :, :m] = M_k.unsqueeze(0).unsqueeze(0) * dSdu_hat[:, :, :m]
        diss = torch.fft.irfft(diss_hat, n=N)
        dudt = rev + diss
        dudt = self._project_energy_conservation(dudt, dEdu)
        dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)
        if self.use_residual:
            gate = torch.sigmoid(self.residual_gate)
            residual = gate * self.residual(u_leaf)
            residual = self._project_energy_conservation(residual, dEdu)
            dudt = dudt + residual
        return u + dudt

    def _project_energy_conservation(self, dudt, dEdu):
        inner_dudt_dE = (dudt * dEdu).sum(dim=-1, keepdim=True)
        norm_dE_sq = (dEdu * dEdu).sum(dim=-1, keepdim=True) + 1e-10
        return dudt - (inner_dudt_dE / norm_dE_sq) * dEdu

    def _ensure_entropy_production(self, dudt, dSdu, dEdu):
        dSdt = (dSdu * dudt).sum(dim=-1, keepdim=True)
        violation = F.relu(-dSdt)
        if violation.sum() > 0:
            inner_SE = (dSdu * dEdu).sum(dim=-1, keepdim=True)
            norm_E_sq = (dEdu * dEdu).sum(dim=-1, keepdim=True) + 1e-10
            dSdu_perp = dSdu - (inner_SE / norm_E_sq) * dEdu
            inner_S_Sperp = (dSdu * dSdu_perp).sum(dim=-1, keepdim=True) + 1e-10
            alpha = violation / inner_S_Sperp
            dudt = dudt + alpha * dSdu_perp
        return dudt

    def predict_with_info(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        u_next = self.forward(u)
        u_next_leaf = u_next.detach().requires_grad_(True)
        E_next = self.E_net(u_next_leaf)
        S_next = self.S_net(u_next_leaf)
        info = {'E': E.detach(), 'S': S.detach(),
                'E_next': E_next.detach(), 'S_next': S_next.detach(),
                'dEdu': dEdu.detach(), 'dSdu': dSdu.detach(),
                'dE': (E_next - E).detach(), 'dS': (S_next - S).detach()}
        return u_next, info

    def degeneracy_loss(self, u):
        """Exact by construction when degeneracy_construction=True, so returns 0."""
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf)
        S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        L_k, M_k = self._get_operators()
        m = L_k.shape[0]
        dEdu_hat = torch.fft.rfft(dEdu, dim=-1)
        dSdu_hat = torch.fft.rfft(dSdu, dim=-1)
        L_dS = L_k.unsqueeze(0).unsqueeze(0) * dSdu_hat[:, :, :m]
        M_dE = M_k.unsqueeze(0).unsqueeze(0) * dEdu_hat[:, :, :m]
        return (L_dS.abs()**2).mean() + (M_dE.abs()**2).mean()


def train_model(model, data_in, data_out, pde_type, model_type='fno',
                n_epochs=150, lr=1e-3, batch_size=32, device='cpu',
                e_sup_mode='none', min_diss_weight=0.0):
    """1D trainer matching the 2D method: NO functional supervision by default."""
    model = model.to(device)
    data_in = data_in.to(device)
    data_out = data_out.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)

    n_samples = data_in.shape[0]

    for epoch in range(n_epochs):
        model.train()
        perm = torch.randperm(n_samples, device=device)
        epoch_losses = defaultdict(float)
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i+batch_size]
            nt = data_in.shape[1]
            t_idx = torch.randint(0, nt, (1,)).item()
            x = data_in[idx, t_idx:t_idx+1, :].clone()
            y = data_out[idx, t_idx:t_idx+1, :].clone()

            if epoch > 30 and torch.rand(1).item() < 0.3:
                t_start = torch.randint(0, max(1, nt-2), (1,)).item()
                x = data_in[idx, t_start:t_start+1, :].clone()
                rollout_len = min(3, nt - t_start)
                pred = x
                rollout_loss = 0
                for step in range(rollout_len):
                    pred = model(pred)
                    target = data_out[idx, t_start+step:t_start+step+1, :]
                    rollout_loss += F.mse_loss(pred, target)
                loss = rollout_loss / rollout_len
                epoch_losses['rollout'] += loss.item()
                if model_type == 'generic':
                    deg = model.degeneracy_loss(x)
                    loss += min(1.0, epoch / 50.0) * 0.01 * deg
                    epoch_losses['degeneracy'] += deg.item()
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); n_batches += 1
                continue

            pred = model(x)
            loss = F.mse_loss(pred, y)
            epoch_losses['data'] += loss.item()

            if model_type == 'ep-fno':
                _, info = model.predict_with_info(x)
                ep = model.energy_penalty(info, pde_type)
                loss += min(1.0, epoch / 30.0) * 0.1 * ep
                epoch_losses['energy_penalty'] += ep.item()
            elif model_type == 'generic':
                deg = model.degeneracy_loss(x)
                loss += min(1.0, epoch / 50.0) * 0.01 * deg
                epoch_losses['degeneracy'] += deg.item()

            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); n_batches += 1

        scheduler.step()
        if (epoch + 1) % 50 == 0 or epoch == 0:
            avg = {k: v / max(n_batches, 1) for k, v in epoch_losses.items()}
            extras = ' '.join(f"{k}={v:.6f}" for k, v in avg.items())
            print(f"  Epoch {epoch+1:3d}: {extras}")

    return model





# ----------------------------------------------------------------------------
# 1D evaluate_model / channel_diagnostics -- verbatim from the repo 1D script
# ----------------------------------------------------------------------------
def evaluate_model(model, data_in, data_out, pde_type, model_type='fno',
                   device='cpu', n_rollout=10):
    """Evaluate a trained model on all metrics.
    Note: no @torch.no_grad() because GENERIC-FNO needs autograd internally."""
    model.eval()
    model = model.to(device)
    data_in = data_in.to(device)
    data_out = data_out.to(device)

    is_time_dep = pde_type in ('heat', 'wave', 'burgers', 'advection')
    n_test = min(50, data_in.shape[0])
    results = {}

    x = data_in[:n_test, 0:1, :]
    y = data_out[:n_test, 0:1, :]
    with torch.enable_grad():
        pred = model(x)
    l2_err = ((pred - y)**2).mean(dim=-1).sqrt() / ((y**2).mean(dim=-1).sqrt() + 1e-8)
    results['l2_single'] = l2_err.mean().item()

    if not is_time_dep:
        results['l2_rollout'] = results['l2_single']
        results['energy_track'] = 0.0
        results['mono_violations'] = 0.0
        results['dE_mean'] = 0.0
        results['dS_mean'] = 0.0
        return results

    nt = min(n_rollout, data_in.shape[1])
    rollout_errors, energy_traj_pred, energy_traj_true = [], [], []
    x_roll = data_in[:n_test, 0:1, :].clone()
    for t in range(nt):
        with torch.enable_grad():
            x_roll = model(x_roll).detach()
        y_t = data_out[:n_test, t:t+1, :]
        err = ((x_roll - y_t)**2).mean(dim=-1).sqrt() / ((y_t**2).mean(dim=-1).sqrt() + 1e-8)
        rollout_errors.append(err.mean().item())
        energy_traj_pred.append(0.5 * (x_roll**2).mean(dim=(-1, -2)))
        energy_traj_true.append(0.5 * (y_t**2).mean(dim=(-1, -2)))
    results['l2_rollout'] = float(np.mean(rollout_errors))

    E_pred_stack = torch.stack(energy_traj_pred, dim=1)
    E_true_stack = torch.stack(energy_traj_true, dim=1)
    results['energy_track'] = ((E_pred_stack - E_true_stack)**2).mean().sqrt().item()

    if pde_type in ('heat', 'burgers'):
        E_init = 0.5 * (data_in[:n_test, 0:1, :]**2).mean(dim=(-1, -2))
        E_all = torch.cat([E_init.unsqueeze(1), E_pred_stack], dim=1)
        dE = E_all[:, 1:] - E_all[:, :-1]
        results['mono_violations'] = (dE > 1e-6).float().mean().item() * 100
    elif pde_type in ('wave', 'advection'):
        E_init = 0.5 * (data_in[:n_test, 0:1, :]**2).mean(dim=(-1, -2))
        E_all = torch.cat([E_init.unsqueeze(1), E_pred_stack], dim=1)
        results['mono_violations'] = E_all.std(dim=1).mean().item()
    else:
        results['mono_violations'] = 0.0

    if model_type == 'generic':
        with torch.enable_grad():
            x_check = data_in[:n_test, 0:1, :].clone()
            _, info = model.predict_with_info(x_check)
            results['dE_mean'] = info['dE'].mean().item()
            results['dS_mean'] = info['dS'].mean().item()
            u_leaf = x_check.detach().requires_grad_(True)
            E = model.E_net(u_leaf)
            dEdu = torch.autograd.grad(E.sum(), u_leaf)[0].detach()
            g = (model(x_check) - x_check).detach()
            num = (dEdu * g).sum(dim=(-1, -2)).abs()
            den = dEdu.flatten(1).norm(dim=1) * g.flatten(1).norm(dim=1) + 1e-12
            results['rE_mean'] = (num / den).mean().item()
    else:
        results['dE_mean'] = results['dS_mean'] = results['rE_mean'] = 0.0
    return results


def channel_diagnostics(model, X, n_batch=40, Y=None):
    """Scale-invariant L-vs-M usage + gauge-invariant dissipation (1D)."""
    device = next(model.parameters()).device
    model.eval()
    X = X[:n_batch].to(device).detach()
    N = X.shape[-1]
    u_leaf = X.clone().requires_grad_(True)
    E = model.E_net(u_leaf); S = model.S_net(u_leaf)
    dEdu = torch.autograd.grad(E.sum(), u_leaf, retain_graph=True)[0].detach()
    dSdu = torch.autograd.grad(S.sum(), u_leaf)[0].detach()
    L_k, M_k = model._get_operators(); L_k = L_k.detach(); M_k = M_k.detach(); m = L_k.shape[0]
    if getattr(model, 'degeneracy_construction', False):
        rev, diss = model._generic_rhs(dEdu, dSdu); rev = rev.detach(); diss = diss.detach()
    else:
        dEh = torch.fft.rfft(dEdu, dim=-1); dSh = torch.fft.rfft(dSdu, dim=-1)
        rev_hat = torch.zeros_like(dEh); diss_hat = torch.zeros_like(dSh)
        rev_hat[:, :, :m] = L_k * dEh[:, :, :m]; diss_hat[:, :, :m] = M_k * dSh[:, :, :m]
        rev = torch.fft.irfft(rev_hat, n=N); diss = torch.fft.irfft(diss_hat, n=N)
    L_mag = (model.a.detach() ** 2).mean().sqrt().item()
    M_mag = ((model.b_real.detach() ** 2 + model.b_imag.detach() ** 2) ** 2).mean().sqrt().item()
    def pnorm(z): return z.flatten(1).norm(dim=1)
    def ip(a, b): return (a * b).flatten(1).sum(dim=1)
    rho_M = (pnorm(diss) / (pnorm(rev) + pnorm(diss) + 1e-12)).mean().item()
    dudt = (model(X) - X).detach()
    r_S = (ip(dSdu, dudt) / (pnorm(dSdu) * pnorm(dudt) + 1e-12)).mean().item()
    r_E = (ip(dEdu, dudt).abs() / (pnorm(dEdu) * pnorm(dudt) + 1e-12)).mean().item()
    r_mech = (-ip(X, dudt) / (pnorm(X) * pnorm(dudt) + 1e-12)).mean().item()
    Qx = (X ** 2).flatten(1).sum(dim=1)
    pi_model = ((Qx - (model(X).detach() ** 2).flatten(1).sum(dim=1)) / (Qx + 1e-12)).mean().item()
    out = {'rho_M': rho_M, 'r_S': r_S, 'r_E': r_E, 'L_mag': L_mag, 'M_mag': M_mag,
           'r_mech': r_mech, 'pi_model': pi_model}
    if Y is not None:
        Yb = Y[:n_batch].to(device).detach()
        out['pi_true'] = ((Qx - (Yb ** 2).flatten(1).sum(dim=1)) / (Qx + 1e-12)).mean().item()
    return out

# ----------------------------------------------------------------------------
# 1D benchmark driver  (repo protocol: 80/20 split, seed once at top, unpaired
# model inits) + the extra GENERIC-noIN model and a checkpoint-name fix.
# ----------------------------------------------------------------------------
def run_benchmark_1d(save_dir=None, nx=64, n_samples=200, nt=20, n_epochs=150,
                     seed=None, pdes=('heat', 'advection', 'burgers', 'wave')):
    if seed is not None:
        torch.manual_seed(seed); np.random.seed(seed)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("=" * 70)
    print(f"1D-FNO no-IN ablation  device={device}  seed={seed}")
    print("=" * 70)
    WIDTH, MODES, N_LAYERS = 32, 16, 4
    WIDTH_FUNC, MODES_FUNC, N_LAYERS_FUNC = 24, 12, 3
    BATCH = 32
    GENS = {'heat': generate_heat_data, 'wave': generate_wave_data,
            'advection': generate_advection_data, 'burgers': generate_burgers_data}
    GENS = {k: GENS[k] for k in pdes}

    def make_generic():
        return GENERIC_FNO(nx=nx, width_func=WIDTH_FUNC, modes_func=MODES_FUNC,
                           n_layers_func=N_LAYERS_FUNC, modes_op=MODES,
                           degeneracy_construction=True)

    all_results = {}
    for pde, gen in GENS.items():
        print(f"\n{'='*50}\nPDE: {pde.upper()}\n{'='*50}")
        di, do, _ = gen(n_samples=n_samples, nx=nx, nt=nt)
        ntr = int(0.8 * n_samples)
        tr_in, te_in, tr_out, te_out = di[:ntr], di[ntr:], do[:ntr], do[ntr:]
        pde_res = {}
        MODELS = [
            ('FNO', 'fno', lambda: VanillaFNO(WIDTH, MODES, N_LAYERS)),
            ('EP-FNO', 'ep-fno', lambda: EP_FNO(WIDTH, MODES, N_LAYERS)),
            ('GENERIC-FNO', 'generic', make_generic),
            ('GENERIC-noIN', 'generic', lambda: strip_instance_norm(make_generic())),
        ]
        for mname, mtype, ctor in MODELS:
            print(f"\n--- {mname} ---")
            model = ctor(); t0 = time.time()
            model = train_model(model, tr_in, tr_out, pde, mtype, n_epochs=n_epochs,
                                lr=1e-3, batch_size=BATCH, device=device,
                                e_sup_mode='none')
            r = evaluate_model(model, te_in, te_out, pde, mtype, device=device, n_rollout=10)
            r['time'] = time.time() - t0; r['params'] = count_params(model)
            pde_res[mname] = r
            print(f"  L2-1step={r['l2_single']:.6f} rollout={r['l2_rollout']:.6f} "
                  f"E-track={r['energy_track']:.6f} Mono={r['mono_violations']:.2f}")
            if mtype == 'generic':
                print(f"  r_E={r['rE_mean']:.2e}  dE/step={r['dE_mean']:.2e} dS/step={r['dS_mean']:.2e}")
                diag = channel_diagnostics(model, te_in[:, :1, :], n_batch=40, Y=te_out[:, :1, :])
                r['diagnostic'] = diag
                print(f"  [diag] rho_M={diag['rho_M']:.4f} r_S={diag['r_S']:.2e} r_E={diag['r_E']:.2e}")
                print(f"  [gauge-inv] r_mech={diag['r_mech']:+.4f} pi_true={diag.get('pi_true', float('nan')):+.4e}")
                if save_dir:
                    os.makedirs(save_dir, exist_ok=True)
                    # checkpoint-name fix: key on model NAME so the two GENERIC
                    # variants do not overwrite each other.
                    tag = 'generic' if mname == 'GENERIC-FNO' else 'generic_noIN'
                    ckpt = os.path.join(save_dir, f'{tag}_fno_1d_{pde}_nx{nx}_seed{seed}.pt')
                    torch.save({'state_dict': model.state_dict(), 'nx': nx, 'pde': pde,
                                'model': mname, 'seed': seed}, ckpt)
                    print(f"  saved checkpoint -> {ckpt}")
            del model
            if device == 'cuda':
                torch.cuda.empty_cache()
        all_results[pde] = pde_res
    return all_results


def run_benchmark_seeds_1d(n_seeds=5, save_dir=None, **kwargs):
    runs = []
    for s in range(n_seeds):
        print("\n" + "#" * 80 + f"\n# 1D SEED {s}/{n_seeds-1}\n" + "#" * 80)
        runs.append(run_benchmark_1d(seed=s, save_dir=(save_dir if s == 0 else None), **kwargs))
    _report_1d(runs, n_seeds)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True); ts = time.strftime('%Y%m%d_%H%M%S')
        with open(os.path.join(save_dir, f'noIN_1d_seeds_{ts}.pkl'), 'wb') as f:
            pickle.dump({'runs': runs, 'n_seeds': n_seeds}, f)
        print(f"\nSaved 1D seed runs -> {save_dir}/noIN_1d_seeds_{ts}.pkl")
    return runs


def _sign_test_p(diffs):
    from math import comb
    d = [x for x in diffs if x != 0]; n = len(d); k = sum(1 for x in d if x < 0)
    if n == 0: return 1.0
    tail = min(k, n - k)
    return min(1.0, 2 * sum(comb(n, i) for i in range(tail + 1)) / 2**n)


def _report_1d(runs, n_seeds):
    pdes = list(runs[0].keys()); models = list(runs[0][pdes[0]].keys())
    ms = lambda vals: (float(np.mean(vals)), float(np.std(vals)))
    get = lambda pde, m, key: [r[pde][m][key] for r in runs]
    print("\n" + "=" * 84)
    print(f"1D-FNO no-IN ablation: {n_seeds}-seed mean +/- std")
    print("=" * 84)
    for pde in pdes:
        print(f"\n{pde.upper()}")
        print(f"  {'Model':<14}{'params':>9}{'L2-1step':>18}{'L2-roll':>18}{'E-track':>12}{'Mono%':>9}")
        for m in models:
            s = ms(get(pde, m, 'l2_single')); q = ms(get(pde, m, 'l2_rollout'))
            et = ms(get(pde, m, 'energy_track')); mo = ms(get(pde, m, 'mono_violations'))
            print(f"  {m:<14}{runs[0][pde][m]['params']:>9,}{s[0]:>9.4f}+/-{s[1]:<6.4f}"
                  f"{q[0]:>9.4f}+/-{q[1]:<6.4f}{et[0]:>12.4f}{mo[0]:>9.2f}")
        # paired: both GENERIC variants vs FNO/EP-FNO, and noIN vs GENERIC-FNO
        for gv in ('GENERIC-FNO', 'GENERIC-noIN'):
            g = np.array(get(pde, gv, 'l2_rollout'))
            for bl in ('FNO', 'EP-FNO'):
                d = g - np.array(get(pde, bl, 'l2_rollout'))
                print(f"    paired {gv:12s}-{bl:7s}: {d.mean():+.4f}  better {int((d<0).sum())}/{n_seeds}"
                      f"  sign p={_sign_test_p(d.tolist()):.3f}")
        d = np.array(get(pde, 'GENERIC-noIN', 'l2_rollout')) - np.array(get(pde, 'GENERIC-FNO', 'l2_rollout'))
        print(f"    paired noIN - GENERIC-FNO   : {d.mean():+.4f}  noIN better {int((d<0).sum())}/{n_seeds}"
              f"  sign p={_sign_test_p(d.tolist()):.3f}")
    # LaTeX (rollout, all four models)
    print("\n% ---- LaTeX (1D-FNO no-IN ablation, rollout L2) ----")
    print(r"\begin{tabular}{l" + "c" * len(models) + "}")
    print(r"\toprule")
    print("PDE & " + " & ".join(models) + r" \\"); print(r"\midrule")
    for pde in pdes:
        cells = {m: ms(get(pde, m, 'l2_rollout')) for m in models}
        best = min(v[0] for v in cells.values())
        row = []
        for m in models:
            c = f"{cells[m][0]:.3f} $\\pm$ {cells[m][1]:.3f}"
            row.append(r"\textbf{" + c + "}" if cells[m][0] == best else c)
        print(f"{pde.capitalize()} & " + " & ".join(row) + r" \\")
    print(r"\bottomrule"); print(r"\end{tabular}")
    print(f"% {n_seeds} seeds; repo 1D protocol (80/20 split, seed once at top).")

# ============================================================================
# ============================  2D  FNO  SECTION  ============================
# ============================================================================
# Classes/generators verbatim from the repo 2D script; drivers suffixed _2d to
# coexist with the 1D ones in this single file.

def _random_field_2d(nx, ny, max_mode, n_modes, amp_scale=0.5, device='cpu'):
    x = torch.linspace(0, 2*math.pi, nx+1, device=device)[:-1]
    y = torch.linspace(0, 2*math.pi, ny+1, device=device)[:-1]
    X, Y = torch.meshgrid(x, y, indexing='ij')
    u = torch.zeros(nx, ny, device=device)
    for _ in range(n_modes):
        kx = torch.randint(1, max_mode, (1,)).item()
        ky = torch.randint(1, max_mode, (1,)).item()
        amp = torch.randn(1).item() * amp_scale
        phase = torch.rand(1).item() * 2 * math.pi
        u += amp * torch.sin(kx * X + ky * Y + phase)
    return u


def generate_heat_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, nu=0.02, device='cpu'):
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    decay = torch.exp(-nu * (KX**2 + KY**2) * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0); traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * decay
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'heat'


def generate_wave_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0, device='cpu'):
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    omega = c * torch.sqrt(KX**2 + KY**2)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, nx//8, n_modes, device=device)
        v0 = _random_field_2d(nx, nx, nx//8, n_modes, amp_scale=0.3, device=device)
        u_hat = torch.fft.rfft2(u0); v_hat = torch.fft.rfft2(v0); traj = [u0.clone()]
        for t in range(nt):
            cos_w = torch.cos(omega * dt); sin_w = torch.sin(omega * dt)
            safe_omega = torch.where(omega > 1e-8, omega, torch.ones_like(omega))
            u_new = cos_w * u_hat + (sin_w / safe_omega) * v_hat
            v_new = -safe_omega * sin_w * u_hat + cos_w * v_hat
            u_hat, v_hat = u_new, v_new
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'wave'


def generate_advection_data_2d(n_samples=150, nx=128, nt=15, dt=0.005, c=1.0,
                               max_mode=6, device='cpu'):
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    phase = torch.exp(-1j * c * (KX + KY) * dt)
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(3, 7, (1,)).item()
        u0 = _random_field_2d(nx, nx, max_mode, n_modes, device=device)
        u_hat = torch.fft.rfft2(u0); traj = [u0.clone()]
        for t in range(nt):
            u_hat = u_hat * phase
            traj.append(torch.fft.irfft2(u_hat, s=(nx, nx)))
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'advection'


def generate_burgers_data_2d(n_samples=150, nx=128, nt=15, dt=0.002, nu=0.02, device='cpu'):
    kx = torch.fft.fftfreq(nx, d=1.0/nx).to(device)
    ky = torch.fft.rfftfreq(nx, d=1.0/nx).to(device)
    KX, KY = torch.meshgrid(kx, ky, indexing='ij')
    k_sq = KX**2 + KY**2
    data_in, data_out = [], []
    for _ in range(n_samples):
        n_modes = torch.randint(2, 5, (1,)).item()
        u = _random_field_2d(nx, nx, 5, n_modes, amp_scale=0.3, device=device)
        traj = [u.clone()]
        for t in range(nt):
            u_hat = torch.fft.rfft2(u) / (1 + nu * k_sq * dt)
            u = torch.fft.irfft2(u_hat, s=(nx, nx))
            ux = torch.fft.irfft2(1j * KX * torch.fft.rfft2(u), s=(nx, nx))
            uy = torch.fft.irfft2(1j * KY * torch.fft.rfft2(u), s=(nx, nx))
            u = u - dt * u * (ux + uy)
            traj.append(u.clone())
        traj = torch.stack(traj, dim=0)
        data_in.append(traj[:-1]); data_out.append(traj[1:])
    return torch.stack(data_in), torch.stack(data_out), 'burgers'


class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1 = modes1; self.modes2 = modes2
        scale = 1.0 / (in_ch * out_ch)
        self.W1 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))
        self.W2 = nn.Parameter(scale * torch.randn(out_ch, in_ch, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        x_hat = torch.fft.rfft2(x, dim=(-2, -1))
        out_hat = torch.zeros(B, self.W1.shape[0], H, W // 2 + 1,
                              dtype=torch.cfloat, device=x.device)
        m1 = min(self.modes1, H // 2); m2 = min(self.modes2, W // 2 + 1)
        out_hat[:, :, :m1, :m2] = torch.einsum('bixy,oixy->boxy', x_hat[:, :, :m1, :m2], self.W1[:, :, :m1, :m2])
        out_hat[:, :, -m1:, :m2] = torch.einsum('bixy,oixy->boxy', x_hat[:, :, -m1:, :m2], self.W2[:, :, :m1, :m2])
        return torch.fft.irfft2(out_hat, s=(H, W))


class FNO_Block2d(nn.Module):
    def __init__(self, width, modes1, modes2):
        super().__init__()
        self.conv = SpectralConv2d(width, width, modes1, modes2)
        self.skip = nn.Conv2d(width, width, 1)
        self.norm = nn.InstanceNorm2d(width)
    def forward(self, x):
        return F.gelu(self.norm(self.conv(x) + self.skip(x)))


class FNO_Backbone2d(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, width=32, modes=16, n_layers=4):
        super().__init__()
        self.lift = nn.Conv2d(in_ch, width, 1)
        self.blocks = nn.ModuleList([FNO_Block2d(width, modes, modes) for _ in range(n_layers)])
        self.proj = nn.Sequential(nn.Conv2d(width, width, 1), nn.GELU(), nn.Conv2d(width, out_ch, 1))
    def forward(self, x):
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.proj(x)


class FunctionalNet2d(nn.Module):
    def __init__(self, width=24, modes=12, n_layers=3):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width, modes=modes, n_layers=n_layers)
        self.head = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, 1))
    def forward(self, u):
        density = self.backbone(u)
        integral = density.mean(dim=(-1, -2))
        return self.head(integral).squeeze(-1)


class VanillaFNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width, modes=modes, n_layers=n_layers)
    def forward(self, u): return u + self.backbone(u)
    def predict_with_info(self, u): return self.forward(u), {}


class EP_FNO2d(nn.Module):
    def __init__(self, width=32, modes=16, n_layers=4):
        super().__init__()
        self.backbone = FNO_Backbone2d(in_ch=1, out_ch=1, width=width, modes=modes, n_layers=n_layers)
    def forward(self, u): return u + self.backbone(u)
    def predict_with_info(self, u):
        u_next = self.forward(u)
        E_in = 0.5 * (u**2).mean(dim=(-1, -2)).mean(dim=-1)
        E_out = 0.5 * (u_next**2).mean(dim=(-1, -2)).mean(dim=-1)
        return u_next, {'dE': E_out - E_in}
    def energy_penalty(self, info, pde_type):
        dE = info['dE']
        if pde_type in ('heat', 'burgers'): return (F.relu(dE)**2).mean()
        elif pde_type in ('wave', 'advection'): return (dE**2).mean()
        return torch.tensor(0.0, device=dE.device)


class GENERIC_FNO2d(nn.Module):
    def __init__(self, nx=128, width_func=24, modes_func=12, n_layers_func=3,
                 modes_op=16, residual_gate_init=-3.0, l2_vargrad=False,
                 use_residual=True, degeneracy_construction=True):
        super().__init__()
        self.nx = nx; self.modes_op = modes_op
        self.degeneracy_construction = degeneracy_construction
        self.l2_vargrad = l2_vargrad; self.use_residual = use_residual
        m1 = min(modes_op, nx // 2); m2 = min(modes_op, nx // 2 + 1)
        self.m1, self.m2 = m1, m2
        self.E_net = FunctionalNet2d(width=width_func, modes=modes_func, n_layers=n_layers_func)
        self.S_net = FunctionalNet2d(width=width_func, modes=modes_func, n_layers=n_layers_func)
        self.a_pos = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.a_neg = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_pos_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_pos_i = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_r = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.b_neg_i = nn.Parameter(0.3 * torch.randn(m1, m2))
        self.residual = nn.Sequential(nn.Conv2d(1, 16, 1), nn.GELU(), nn.Conv2d(16, 1, 1))
        self.residual_gate = nn.Parameter(torch.tensor(float(residual_gate_init)))

    def _apply_operators(self, dEdu_hat, dSdu_hat, H, W):
        m1, m2 = self.m1, self.m2
        rev_hat = torch.zeros_like(dEdu_hat); diss_hat = torch.zeros_like(dSdu_hat)
        rev_hat[:, :, :m1, :m2] = 1j * self.a_pos * dEdu_hat[:, :, :m1, :m2]
        rev_hat[:, :, -m1:, :m2] = 1j * self.a_neg * dEdu_hat[:, :, -m1:, :m2]
        M_pos = self.b_pos_r**2 + self.b_pos_i**2; M_neg = self.b_neg_r**2 + self.b_neg_i**2
        diss_hat[:, :, :m1, :m2] = M_pos * dSdu_hat[:, :, :m1, :m2]
        diss_hat[:, :, -m1:, :m2] = M_neg * dSdu_hat[:, :, -m1:, :m2]
        return rev_hat, diss_hat

    def _L_apply(self, v, H, W):
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1)); out = torch.zeros_like(vh)
        out[:, :, :m1, :m2] = 1j * self.a_pos * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = 1j * self.a_neg * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    def _M_apply(self, v, H, W):
        m1, m2 = self.m1, self.m2
        vh = torch.fft.rfft2(v, dim=(-2, -1)); out = torch.zeros_like(vh)
        Mp = self.b_pos_r**2 + self.b_pos_i**2; Mn = self.b_neg_r**2 + self.b_neg_i**2
        out[:, :, :m1, :m2] = Mp * vh[:, :, :m1, :m2]
        out[:, :, -m1:, :m2] = Mn * vh[:, :, -m1:, :m2]
        return torch.fft.irfft2(out, s=(H, W))

    @staticmethod
    def _remove(v, w):
        ip = (v * w).sum(dim=(-1, -2), keepdim=True)
        nn_ = (w * w).sum(dim=(-1, -2), keepdim=True) + 1e-12
        return v - (ip / nn_) * w

    def _generic_rhs(self, dEdu, dSdu, H, W):
        rev = self._remove(self._L_apply(self._remove(dEdu, dSdu), H, W), dSdu)
        diss = self._remove(self._M_apply(self._remove(dSdu, dEdu), H, W), dEdu)
        return rev, diss

    def forward(self, u):
        B, C, H, W = u.shape
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        if self.l2_vargrad:
            scale = (H * W) / (2.0 * math.pi) ** 2; dEdu = dEdu * scale; dSdu = dSdu * scale
        if self.degeneracy_construction:
            rev, diss = self._generic_rhs(dEdu, dSdu, H, W)
            return u + rev + diss
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1)); dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))
        rev_hat, diss_hat = self._apply_operators(dEdu_hat, dSdu_hat, H, W)
        rev = torch.fft.irfft2(rev_hat, s=(H, W)); diss = torch.fft.irfft2(diss_hat, s=(H, W))
        dudt = rev + diss
        dudt = self._project_energy_conservation(dudt, dEdu)
        dudt = self._ensure_entropy_production(dudt, dSdu, dEdu)
        if self.use_residual:
            gate = torch.sigmoid(self.residual_gate)
            residual = gate * self.residual(u_leaf)
            residual = self._project_energy_conservation(residual, dEdu)
            dudt = dudt + residual
        return u + dudt

    def _project_energy_conservation(self, dudt, dEdu):
        inner = (dudt * dEdu).sum(dim=(-1, -2), keepdim=True)
        norm_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
        return dudt - (inner / norm_sq) * dEdu

    def _ensure_entropy_production(self, dudt, dSdu, dEdu):
        dSdt = (dSdu * dudt).sum(dim=(-1, -2), keepdim=True)
        violation = F.relu(-dSdt)
        if violation.sum() > 0:
            inner_SE = (dSdu * dEdu).sum(dim=(-1, -2), keepdim=True)
            norm_E_sq = (dEdu * dEdu).sum(dim=(-1, -2), keepdim=True) + 1e-10
            dSdu_perp = dSdu - (inner_SE / norm_E_sq) * dEdu
            inner_S_Sperp = (dSdu * dSdu_perp).sum(dim=(-1, -2), keepdim=True) + 1e-10
            alpha = violation / inner_S_Sperp
            dudt = dudt + alpha * dSdu_perp
        return dudt

    def predict_with_info(self, u):
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        _ = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        _ = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        u_next = self.forward(u)
        u_next_leaf = u_next.detach().requires_grad_(True)
        E_next = self.E_net(u_next_leaf); S_next = self.S_net(u_next_leaf)
        return u_next, {'E': E.detach(), 'S': S.detach(),
                        'dE': (E_next - E).detach(), 'dS': (S_next - S).detach()}

    def degeneracy_loss(self, u):
        if self.degeneracy_construction:
            return torch.zeros((), device=u.device)
        u_leaf = u.detach().requires_grad_(True)
        E = self.E_net(u_leaf); S = self.S_net(u_leaf)
        dEdu = torch.autograd.grad(E.sum(), u_leaf, create_graph=True)[0]
        dSdu = torch.autograd.grad(S.sum(), u_leaf, create_graph=True)[0]
        dEdu_hat = torch.fft.rfft2(dEdu, dim=(-2, -1)); dSdu_hat = torch.fft.rfft2(dSdu, dim=(-2, -1))
        m1, m2 = self.m1, self.m2
        L_dS_pos = 1j * self.a_pos * dSdu_hat[:, :, :m1, :m2]
        L_dS_neg = 1j * self.a_neg * dSdu_hat[:, :, -m1:, :m2]
        loss_L = (L_dS_pos.abs()**2).mean() + (L_dS_neg.abs()**2).mean()
        M_pos = self.b_pos_r**2 + self.b_pos_i**2; M_neg = self.b_neg_r**2 + self.b_neg_i**2
        M_dE_pos = M_pos * dEdu_hat[:, :, :m1, :m2]; M_dE_neg = M_neg * dEdu_hat[:, :, -m1:, :m2]
        loss_M = (M_dE_pos.abs()**2).mean() + (M_dE_neg.abs()**2).mean()
        return loss_L + loss_M

# ----------------------------------------------------------------------------
# 2D train / evaluate / channel_diagnostics -- verbatim from the repo 2D script
# (suffixed _2d).  train_model_2d default e_sup_mode='none'.
# ----------------------------------------------------------------------------
def evaluate_model_2d(model, data_in, data_out, pde_type, model_type='fno',
                      device='cpu', n_rollout=10, eval_batch=4):
    model.eval(); model = model.to(device)
    n_test = min(40, data_in.shape[0]); nt = min(n_rollout, data_in.shape[1])
    is_generic = (model_type == 'generic')
    l2_single = []; rollout_err_per_t = [[] for _ in range(nt)]
    E_pred_per_t = [[] for _ in range(nt)]; E_true_per_t = [[] for _ in range(nt)]
    E_init_list, dE_list, dS_list = [], [], []
    for start in range(0, n_test, eval_batch):
        end = min(start + eval_batch, n_test)
        x0 = data_in[start:end, 0:1, :, :].to(device); y0 = data_out[start:end, 0:1, :, :].to(device)
        with torch.enable_grad():
            pred = model(x0).detach()
        l2 = ((pred - y0)**2).mean(dim=(-1, -2)).sqrt() / ((y0**2).mean(dim=(-1, -2)).sqrt() + 1e-8)
        l2_single.append(l2.flatten().cpu())
        if is_generic:
            with torch.enable_grad():
                _, info = model.predict_with_info(x0)
            dE_list.append(info['dE'].flatten().cpu()); dS_list.append(info['dS'].flatten().cpu())
        E_init_list.append((0.5 * (x0**2).mean(dim=(-1, -2, -3))).cpu())
        x_roll = x0.clone()
        for t in range(nt):
            with torch.enable_grad():
                x_roll = model(x_roll).detach()
            y_t = data_out[start:end, t:t+1, :, :].to(device)
            err = ((x_roll - y_t)**2).mean(dim=(-1, -2)).sqrt() / ((y_t**2).mean(dim=(-1, -2)).sqrt() + 1e-8)
            rollout_err_per_t[t].append(err.flatten().cpu())
            E_pred_per_t[t].append((0.5 * (x_roll**2).mean(dim=(-1, -2, -3))).cpu())
            E_true_per_t[t].append((0.5 * (y_t**2).mean(dim=(-1, -2, -3))).cpu())
            del y_t
        del x0, y0, x_roll, pred
        if device == 'cuda': torch.cuda.empty_cache()
    results = {}
    results['l2_single'] = torch.cat(l2_single).mean().item()
    results['l2_rollout'] = float(np.mean([torch.cat(r).mean().item() for r in rollout_err_per_t]))
    E_pred_stack = torch.stack([torch.cat(e) for e in E_pred_per_t], dim=1)
    E_true_stack = torch.stack([torch.cat(e) for e in E_true_per_t], dim=1)
    results['energy_track'] = ((E_pred_stack - E_true_stack)**2).mean().sqrt().item()
    E_init = torch.cat(E_init_list)
    if pde_type in ('heat', 'burgers'):
        E_all = torch.cat([E_init.unsqueeze(1), E_pred_stack], dim=1)
        dE = E_all[:, 1:] - E_all[:, :-1]
        results['mono_violations'] = (dE > 1e-6).float().mean().item() * 100
    elif pde_type in ('wave', 'advection'):
        E_all = torch.cat([E_init.unsqueeze(1), E_pred_stack], dim=1)
        results['mono_violations'] = E_all.std(dim=1).mean().item()
    else:
        results['mono_violations'] = 0.0
    if is_generic:
        results['dE_mean'] = torch.cat(dE_list).mean().item()
        results['dS_mean'] = torch.cat(dS_list).mean().item()
    else:
        results['dE_mean'] = results['dS_mean'] = 0.0
    return results


def train_model_2d(model, data_in, data_out, pde_type, model_type='fno',
                   n_epochs=120, lr=1e-3, batch_size=16, device='cpu',
                   degeneracy_weight=0.01):
    model = model.to(device); data_in = data_in.to(device); data_out = data_out.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)
    n_samples = data_in.shape[0]; nt = data_in.shape[1]
    for epoch in range(n_epochs):
        model.train(); perm = torch.randperm(n_samples, device=device)
        epoch_losses = defaultdict(float); n_batches = 0
        for i in range(0, n_samples, batch_size):
            idx = perm[i:i+batch_size]
            t_idx = torch.randint(0, nt, (1,)).item()
            x = data_in[idx, t_idx:t_idx+1, :, :].clone(); y = data_out[idx, t_idx:t_idx+1, :, :].clone()
            if epoch > 25 and torch.rand(1).item() < 0.25:
                t_start = torch.randint(0, max(1, nt-2), (1,)).item()
                x = data_in[idx, t_start:t_start+1, :, :].clone()
                rollout_len = min(2, nt - t_start); pred = x; rollout_loss = 0
                for step in range(rollout_len):
                    pred = model(pred)
                    rollout_loss += F.mse_loss(pred, data_out[idx, t_start+step:t_start+step+1, :, :])
                loss = rollout_loss / rollout_len; epoch_losses['rollout'] += loss.item()
                if model_type == 'generic':
                    deg = model.degeneracy_loss(x); loss += min(1.0, epoch / 50.0) * degeneracy_weight * deg
                    epoch_losses['degeneracy'] += deg.item()
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step(); n_batches += 1; continue
            pred = model(x); loss = F.mse_loss(pred, y); epoch_losses['data'] += loss.item()
            if model_type == 'ep-fno':
                _, info = model.predict_with_info(x); ep = model.energy_penalty(info, pde_type)
                loss += min(1.0, epoch / 30.0) * 0.1 * ep; epoch_losses['energy_penalty'] += ep.item()
            elif model_type == 'generic':
                deg = model.degeneracy_loss(x); loss += min(1.0, epoch / 50.0) * degeneracy_weight * deg
                epoch_losses['degeneracy'] += deg.item()
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step(); n_batches += 1
        scheduler.step()
        if (epoch + 1) % 30 == 0 or epoch == 0:
            avg = {k: v / max(n_batches, 1) for k, v in epoch_losses.items()}
            print("  Epoch %3d: %s" % (epoch+1, ' '.join(f"{k}={v:.6f}" for k, v in avg.items())))
    return model


def channel_diagnostics_2d(model, X, n_batch=8, Y=None):
    device = next(model.parameters()).device
    model.eval(); X = X[:n_batch].to(device).detach(); H, W = X.shape[-2], X.shape[-1]
    u_leaf = X.clone().requires_grad_(True)
    E = model.E_net(u_leaf); S = model.S_net(u_leaf)
    dEdu = torch.autograd.grad(E.sum(), u_leaf, retain_graph=True)[0].detach()
    dSdu = torch.autograd.grad(S.sum(), u_leaf)[0].detach()
    if getattr(model, 'degeneracy_construction', False):
        rev, diss = model._generic_rhs(dEdu, dSdu, H, W); rev = rev.detach(); diss = diss.detach()
    else:
        dEh = torch.fft.rfft2(dEdu, dim=(-2, -1)); dSh = torch.fft.rfft2(dSdu, dim=(-2, -1))
        rev_hat, diss_hat = model._apply_operators(dEh, dSh, H, W)
        rev = torch.fft.irfft2(rev_hat.detach(), s=(H, W)); diss = torch.fft.irfft2(diss_hat.detach(), s=(H, W))
    def pnorm(z): return z.flatten(1).norm(dim=1)
    def ip(a, b): return (a * b).flatten(1).sum(dim=1)
    rho_M = (pnorm(diss) / (pnorm(rev) + pnorm(diss) + 1e-12)).mean().item()
    Xn = model(X).detach(); dudt = (Xn - X).detach()
    r_S = (ip(dSdu, dudt) / (pnorm(dSdu) * pnorm(dudt) + 1e-12)).mean().item()
    r_E = (ip(dEdu, dudt).abs() / (pnorm(dEdu) * pnorm(dudt) + 1e-12)).mean().item()
    r_mech = (-ip(X, dudt) / (pnorm(X) * pnorm(dudt) + 1e-12)).mean().item()
    Qx = (X ** 2).flatten(1).sum(dim=1)
    out = {'rho_M': rho_M, 'r_S': r_S, 'r_E': r_E, 'r_mech': r_mech}
    if Y is not None:
        Yb = Y[:n_batch].to(device).detach()
        out['pi_true'] = ((Qx - (Yb ** 2).flatten(1).sum(dim=1)) / (Qx + 1e-12)).mean().item()
    return out


def run_benchmark_2d(save_dir=None, nx=128, n_samples=150, nt=15, n_epochs=120,
                     seed=None, pdes=('heat', 'advection', 'burgers')):
    if seed is not None:
        torch.manual_seed(seed); np.random.seed(seed)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print("=" * 70)
    print(f"2D-FNO no-IN ablation  device={device}  nx={nx}  seed={seed}")
    print("=" * 70)
    WIDTH, MODES, N_LAYERS = 32, 16, 4
    WIDTH_FUNC, MODES_FUNC, N_LAYERS_FUNC = 24, 12, 3
    BATCH = 16
    GENS = {'heat': generate_heat_data_2d, 'wave': generate_wave_data_2d,
            'advection': generate_advection_data_2d, 'burgers': generate_burgers_data_2d}
    GENS = {k: GENS[k] for k in pdes}

    def make_generic():
        return GENERIC_FNO2d(nx, WIDTH_FUNC, MODES_FUNC, N_LAYERS_FUNC, MODES,
                             degeneracy_construction=True)

    all_results = {}
    for pde, gen in GENS.items():
        print(f"\n{'='*50}\nPDE: {pde.upper()}\n{'='*50}")
        di, do, _ = gen(n_samples, nx, nt)
        ntr = int(0.8 * n_samples)
        tr_in, te_in, tr_out, te_out = di[:ntr], di[ntr:], do[:ntr], do[ntr:]
        pde_res = {}
        MODELS = [
            ('FNO', 'fno', lambda: VanillaFNO2d(WIDTH, MODES, N_LAYERS)),
            ('EP-FNO', 'ep-fno', lambda: EP_FNO2d(WIDTH, MODES, N_LAYERS)),
            ('GENERIC-FNO', 'generic', make_generic),
            ('GENERIC-noIN', 'generic', lambda: strip_instance_norm(make_generic())),
        ]
        for mname, mtype, ctor in MODELS:
            print(f"\n--- {mname} ---")
            model = ctor(); t0 = time.time()
            model = train_model_2d(model, tr_in, tr_out, pde, mtype, n_epochs=n_epochs,
                                   lr=1e-3, batch_size=BATCH, device=device)
            r = evaluate_model_2d(model, te_in, te_out, pde, mtype, device=device,
                                  n_rollout=10, eval_batch=4)
            r['time'] = time.time() - t0; r['params'] = count_params(model)
            pde_res[mname] = r
            print(f"  L2-1step={r['l2_single']:.6f} rollout={r['l2_rollout']:.6f} "
                  f"E-track={r['energy_track']:.6f} Mono={r['mono_violations']:.2f} ({r['time']:.1f}s)")
            if mtype == 'generic':
                print(f"  dE/step={r['dE_mean']:.2e} dS/step={r['dS_mean']:.2e}")
                diag = channel_diagnostics_2d(model, te_in[:, :1, :, :], n_batch=8, Y=te_out[:, :1, :, :])
                r['diagnostic'] = diag
                print(f"  [diag] rho_M={diag['rho_M']:.4f} r_E={diag['r_E']:.2e}")
                print(f"  [gauge-inv] r_mech={diag['r_mech']:+.4f} pi_true={diag.get('pi_true', float('nan')):+.4e}")
                if save_dir:
                    os.makedirs(save_dir, exist_ok=True)
                    tag = 'generic' if mname == 'GENERIC-FNO' else 'generic_noIN'
                    ckpt = os.path.join(save_dir, f'{tag}_fno_2d_{pde}_nx{nx}_seed{seed}.pt')
                    torch.save({'state_dict': model.state_dict(), 'nx': nx, 'pde': pde,
                                'model': mname, 'seed': seed}, ckpt)
                    print(f"  saved checkpoint -> {ckpt}")
            del model
            if device == 'cuda': torch.cuda.empty_cache()
        all_results[pde] = pde_res
    return all_results


def run_benchmark_seeds_2d(n_seeds=3, save_dir=None, **kwargs):
    runs = []
    for s in range(n_seeds):
        print("\n" + "#" * 80 + f"\n# 2D SEED {s}/{n_seeds-1}\n" + "#" * 80)
        runs.append(run_benchmark_2d(seed=s, save_dir=(save_dir if s == 0 else None), **kwargs))
    _report_2d(runs, n_seeds)
    if save_dir:
        os.makedirs(save_dir, exist_ok=True); ts = time.strftime('%Y%m%d_%H%M%S')
        with open(os.path.join(save_dir, f'noIN_2d_seeds_{ts}.pkl'), 'wb') as f:
            pickle.dump({'runs': runs, 'n_seeds': n_seeds}, f)
        print(f"\nSaved 2D seed runs -> {save_dir}/noIN_2d_seeds_{ts}.pkl")
    return runs


def _report_2d(runs, n_seeds):
    pdes = list(runs[0].keys()); models = list(runs[0][pdes[0]].keys())
    ms = lambda vals: (float(np.mean(vals)), float(np.std(vals)))
    get = lambda pde, m, key: [r[pde][m][key] for r in runs]
    getd = lambda pde, m, key: [r[pde][m]['diagnostic'][key] for r in runs]
    print("\n" + "=" * 84)
    print(f"2D-FNO no-IN ablation: {n_seeds}-seed mean +/- std")
    print("=" * 84)
    for pde in pdes:
        print(f"\n{pde.upper()}")
        print(f"  {'Model':<14}{'params':>10}{'L2-1step':>18}{'L2-roll':>18}{'E-track':>12}{'Mono%':>9}")
        for m in models:
            s = ms(get(pde, m, 'l2_single')); q = ms(get(pde, m, 'l2_rollout'))
            et = ms(get(pde, m, 'energy_track')); mo = ms(get(pde, m, 'mono_violations'))
            print(f"  {m:<14}{runs[0][pde][m]['params']:>10,}{s[0]:>9.4f}+/-{s[1]:<6.4f}"
                  f"{q[0]:>9.4f}+/-{q[1]:<6.4f}{et[0]:>12.4f}{mo[0]:>9.2f}")
        d = np.array(get(pde, 'GENERIC-noIN', 'l2_rollout')) - np.array(get(pde, 'GENERIC-FNO', 'l2_rollout'))
        print(f"    paired noIN - GENERIC-FNO: {d.mean():+.4f}  noIN better {int((d<0).sum())}/{n_seeds}")
    print("\nGAUGE-INVARIANT DISSIPATION (r_mech mean +/- std)")
    print(f"  {'PDE':<10}{'GENERIC-FNO':>22}{'GENERIC-noIN':>22}{'pi_true':>14}")
    for pde in pdes:
        gi = ms(getd(pde, 'GENERIC-FNO', 'r_mech')); gn = ms(getd(pde, 'GENERIC-noIN', 'r_mech'))
        pt = ms(getd(pde, 'GENERIC-FNO', 'pi_true'))
        print(f"  {pde:<10}{gi[0]:>+11.3f}+/-{gi[1]:<8.3f}{gn[0]:>+11.3f}+/-{gn[1]:<8.3f}{pt[0]:>+14.2e}")
    print("\n% ---- LaTeX (2D-FNO no-IN ablation, rollout L2) ----")
    print(r"\begin{tabular}{l" + "c" * len(models) + "}")
    print(r"\toprule"); print("PDE & " + " & ".join(models) + r" \\"); print(r"\midrule")
    for pde in pdes:
        cells = {m: ms(get(pde, m, 'l2_rollout')) for m in models}
        best = min(v[0] for v in cells.values()); row = []
        for m in models:
            c = f"{cells[m][0]:.3f} $\\pm$ {cells[m][1]:.3f}"
            row.append(r"\textbf{" + c + "}" if cells[m][0] == best else c)
        print(f"{pde.capitalize()} & " + " & ".join(row) + r" \\")
    print(r"\bottomrule"); print(r"\end{tabular}")
    print(f"% {n_seeds} seeds; repo 2D protocol.")


# ============================================================================
# DRIVER
# ============================================================================
if __name__ == '__main__':
    try:
        from google.colab import drive; drive.mount('/content/drive')
        DRIVE = '/content/drive/MyDrive/GENERIC_FNO_results'
    except Exception:
        DRIVE = './GENERIC_FNO_results'
    os.makedirs(DRIVE, exist_ok=True)
    print(f"Results -> {DRIVE}")

    which = os.environ.get('R_RUN', 'both')
    n1 = int(os.environ.get('R_1D_SEEDS', 5))
    n2 = int(os.environ.get('R_2D_SEEDS', 3))
    nx2 = int(os.environ.get('R_2D_NX', 128))
    ep2 = int(os.environ.get('R_2D_EPOCHS', 120))

    if which in ('1d', 'both'):
        print("\n" + "*" * 80 + "\n* 1D-FNO no-IN ablation (5 seeds, 4 PDEs)\n" + "*" * 80)
        run_benchmark_seeds_1d(n_seeds=n1, save_dir=DRIVE,
                               pdes=('heat', 'advection', 'burgers', 'wave'))
    if which in ('2d', 'both'):
        print("\n" + "*" * 80 + "\n* 2D-FNO no-IN ablation (3 seeds, heat/advection/burgers)\n" + "*" * 80)
        run_benchmark_seeds_2d(n_seeds=n2, save_dir=DRIVE, nx=nx2, n_epochs=ep2,
                               pdes=('heat', 'advection', 'burgers'))

Mounted at /content/drive
Results -> /content/drive/MyDrive/GENERIC_FNO_results

********************************************************************************
* 1D-FNO no-IN ablation (5 seeds, 4 PDEs)
********************************************************************************

################################################################################
# 1D SEED 0/4
################################################################################
1D-FNO no-IN ablation  device=cuda  seed=0

PDE: HEAT

--- FNO ---
  Epoch   1: data=0.017076
  Epoch  50: data=0.000071 rollout=0.000105
  Epoch 100: data=0.000004 rollout=0.000011
  Epoch 150: data=0.000006
  L2-1step=0.019207 rollout=0.095538 E-track=0.017949 Mono=7.50

--- EP-FNO ---
  Epoch   1: data=0.003233 energy_penalty=0.000176
  Epoch  50: data=0.000035 energy_penalty=0.000000 rollout=0.000072
  Epoch 100: data=0.000002 energy_penalty=0.000000 rollout=0.000009
  Epoch 150: rollout=0.000008 data=0.000002 energy_penalty=0.0